In [ ]:
CONFIG_NAME = "wav2vec2.yaml"
AUDIO_PATH = "acoustic/data/sound.wav"

In [ ]:
import sys
from pathlib import Path
import torch
import librosa
import logging

sys.path.insert(0, str(Path.cwd()))

from acoustic.utils.config import load_config
from acoustic.models.load_model import build_model
from acoustic.models import get_generate_method

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
config_path = f"acoustic/configs/{CONFIG_NAME}"
cfg = load_config(config_path)

In [ ]:
output_dir = Path(cfg['training']['output_dir'])
final_dir = output_dir / "final_model"

if final_dir.exists():
    checkpoint_dir = final_dir
    logger.info("Using final model")
else:
    checkpoints = sorted(output_dir.glob("checkpoint-*"))
    if not checkpoints:
        raise FileNotFoundError(
            f"No checkpoints or final model found in {output_dir}. Train the model first."
        )
    checkpoint_dir = max(checkpoints, key=lambda p: int(p.name.split("-")[-1]))
    logger.info(f"Using latest checkpoint: {checkpoint_dir}")

model, processor, _ = build_model(cfg)
model = model.from_pretrained(str(checkpoint_dir))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

In [ ]:
default_audio = AUDIO_PATH
test_cfg = cfg.get('test', {})
audio_path = test_cfg.get('audio_path', None)

if audio_path is None:
    print(f"No 'test.audio_path' found in config. Using default: {default_audio}")
    audio_path = default_audio

try:
    audio_array, sr = librosa.load(audio_path, sr=16000, mono=True)
except Exception as e:
    logger.error(f"Failed to load audio from '{audio_path}': {e}")
    raise

In [ ]:
inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
input_key = "input_features" if "input_features" in inputs else "input_values"
input_data = inputs[input_key].to(device)

builder_key = cfg['model']['builder']
generate_fn = get_generate_method(builder_key)

with torch.no_grad():
    predicted_ids = generate_fn(model, input_data, processor)

transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
print("\nРаспознанный текст:", transcription)